<a href="https://colab.research.google.com/github/BilodeauS/TheWalkers_ClassProject_TheWalkers-Group6-/blob/main/consolidate_raw_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mne

from google.colab import files
uploaded = files.upload()   # pick sub01_01_EC.set from your computer (74 MB, takes a minute)

In [ ]:
psd, freqs = spectrum.get_data(return_freqs=True)   # shape: (61 channels, frequencies)

theta = psd[:, (freqs >= 4) & (freqs < 8)].mean()
alpha = psd[:, (freqs >= 8) & (freqs < 13)].mean()

print(f"Theta power (4–8 Hz): {theta:.3e}")
print(f"Alpha power (8–13 Hz): {alpha:.3e}")
print(f"Theta/alpha ratio:     {theta / alpha:.3f}")

In [ ]:
# ============================================================
# Build one combined dataset for the sleepiness EEG project
# OpenNeuro ds004148 — eyes-closed preprocessed EEG + questionnaires
# Paste into ONE Colab cell and run. Takes ~30–60 min the first time.
# ============================================================

import os
import urllib.parse
import urllib.request
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import welch

# ---------- Settings ----------
BASE = "https://s3.amazonaws.com/openneuro.org/ds004148/"
EEG_FOLDER = "derivatives/preprocessed data/preprocessed_data/"
WORK_DIR = "/content/eeg_tmp"          # temporary download spot
OUT_EEG = "/content/eeg_bandpower.csv"  # EEG results (saved as it goes)
OUT_FINAL = "/content/sleepiness_eeg_dataset.csv"  # the final combined file

SUBJECTS = range(1, 61)   # sub01 ... sub60
SESSIONS = [1, 2, 3]
THETA = (4, 8)
ALPHA = (8, 13)

os.makedirs(WORK_DIR, exist_ok=True)


def download(rel_path, dest):
    """Download one file from OpenNeuro's public storage."""
    url = BASE + urllib.parse.quote(rel_path)
    urllib.request.urlretrieve(url, dest)


def band_power(file_path):
    """Load one EEGLAB .set file and return theta, alpha and their ratio."""
    m = sio.loadmat(file_path, squeeze_me=True, struct_as_record=False)
    data = np.asarray(m["data"], dtype=float)   # channels x samples, in microvolts
    if data.ndim == 3:                          # if ever stored as epochs, join them
        data = data.reshape(data.shape[0], -1)
    fs = float(m["srate"])

    # Welch power spectrum with 4-second windows (same as the dataset paper)
    freqs, psd = welch(data, fs=fs, nperseg=int(4 * fs))

    theta = psd[:, (freqs >= THETA[0]) & (freqs < THETA[1])].mean()
    alpha = psd[:, (freqs >= ALPHA[0]) & (freqs < ALPHA[1])].mean()
    return theta, alpha, theta / alpha


# ---------- Step 1: EEG band power for all 180 eyes-closed files ----------
# Resumes where it left off if Colab disconnects.
if os.path.exists(OUT_EEG):
    eeg = pd.read_csv(OUT_EEG)
    done = set(zip(eeg["participant_id"], eeg["session"]))
    rows = eeg.to_dict("records")
else:
    done, rows = set(), []

for s in SUBJECTS:
    for ses in SESSIONS:
        pid = f"sub-{s:02d}"
        if (pid, ses) in done:
            continue
        fname = f"sub{s:02d}_{ses:02d}_EC.set"
        local = os.path.join(WORK_DIR, fname)
        try:
            download(EEG_FOLDER + fname, local)
            theta, alpha, ratio = band_power(local)
            rows.append({"participant_id": pid, "session": ses,
                         "theta": theta, "alpha": alpha, "theta_alpha_ratio": ratio})
            print(f"{fname}: theta={theta:.3f}  alpha={alpha:.3f}  ratio={ratio:.3f}")
        except Exception as e:
            print(f"{fname}: FAILED ({e})")
        finally:
            if os.path.exists(local):
                os.remove(local)   # delete after use so Colab doesn't run out of disk
        pd.DataFrame(rows).to_csv(OUT_EEG, index=False)   # save progress

eeg = pd.DataFrame(rows)
print(f"\nEEG done: {len(eeg)} of 180 recordings")


# ---------- Step 2: questionnaire scores ----------
download("participants.tsv", "/content/participants.tsv")
p = pd.read_csv("/content/participants.tsv", sep="\t")

# some cells say "n/a" -> turn them into blanks so the maths works
num = lambda col: pd.to_numeric(p[col], errors="coerce")
p["ESS_avg"] = pd.concat([num("ESS_1stVisit"), num("ESS_2ndVisit")], axis=1).mean(axis=1)

# reshape: one row per person per session
long_rows = []
for _, r in p.iterrows():
    for ses in SESSIONS:
        long_rows.append({
            "participant_id": r["participant_id"],
            "session": ses,
            "age": r["age"],
            "sex": r["sex"],
            "KSS": pd.to_numeric(r[f"KSS_session{ses}"], errors="coerce"),
            "ARSQ_sleepiness": pd.to_numeric(r[f"(5)Sleepiness_session{ses}"], errors="coerce"),
            "ESS_avg": r["ESS_avg"],
        })
q = pd.DataFrame(long_rows)


# ---------- Step 3: merge into ONE file ----------
final = q.merge(eeg, on=["participant_id", "session"], how="left")
final["log_theta"] = np.log10(final["theta"])
final["log_alpha"] = np.log10(final["alpha"])
final.to_csv(OUT_FINAL, index=False)

print(f"\nSaved {OUT_FINAL}  ({len(final)} rows)")
print(final.head(9))
print("\nMissing values per column:\n", final.isna().sum())

# Download it to your computer
from google.colab import files
files.download(OUT_FINAL)

sub01_01_EC.set: theta=1.164  alpha=2.037  ratio=0.571
sub01_02_EC.set: theta=1.340  alpha=2.112  ratio=0.634
sub01_03_EC.set: theta=1.468  alpha=1.424  ratio=1.031
sub02_01_EC.set: theta=5.627  alpha=30.151  ratio=0.187
sub02_02_EC.set: theta=8.528  alpha=23.932  ratio=0.356
sub02_03_EC.set: theta=9.136  alpha=6.758  ratio=1.352
sub03_01_EC.set: theta=5.347  alpha=7.128  ratio=0.750
sub03_02_EC.set: theta=10.180  alpha=10.789  ratio=0.944
sub03_03_EC.set: theta=4.547  alpha=5.469  ratio=0.831
sub04_01_EC.set: theta=5.467  alpha=19.235  ratio=0.284
sub04_02_EC.set: theta=4.092  alpha=26.933  ratio=0.152
sub04_03_EC.set: theta=3.315  alpha=37.490  ratio=0.088
sub05_01_EC.set: theta=3.781  alpha=7.461  ratio=0.507
sub05_02_EC.set: theta=3.623  alpha=7.396  ratio=0.490
sub05_03_EC.set: theta=3.369  alpha=6.807  ratio=0.495
sub06_01_EC.set: theta=4.091  alpha=6.005  ratio=0.681
sub06_02_EC.set: theta=3.610  alpha=9.660  ratio=0.374
sub06_03_EC.set: theta=4.425  alpha=5.312  ratio=0.833
sub

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>